# P2.2 — Simulasi Konvolusi Unit Hydrograph (UH)
### Studio Pemrograman Python | Teknik Sipil

---

**Skenario:**  
DAS Progo seluas **250 km²** diterjang hujan deras selama 7 jam. Tim Anda diminta
menghitung **hydrograph banjir** yang tiba di outlet sungai menggunakan metode
Unit Hydrograph (UH). Hasilnya akan digunakan untuk perencanaan kapasitas tanggul.

---

## Teori: Konvolusi Unit Hydrograph

DAS bertindak sebagai **filter linier**: input hujan efektif → output debit sungai.

$$Q_t = \sum_{i=1}^{n} U_i \cdot P_{t-i+1}$$

| Simbol | Keterangan | Satuan |
|--------|-----------|--------|
| $Q_t$  | Debit limpasan pada waktu ke-$t$ | m³/s |
| $U_i$  | Ordinat Unit Hydrograph ke-$i$ | m³/s/mm |
| $P_j$  | Hujan efektif pada jam ke-$j$ | mm |

**Ilustrasi matriks konvolusi** (P = [P1,P2,P3], U = [U1,U2,U3]):

| Jam | Dari P1 | Dari P2 | Dari P3 | **Total Q** |
|-----|---------|---------|---------|-------------|
| 1 | P1·U1 | — | — | P1·U1 |
| 2 | P1·U2 | P2·U1 | — | P1·U2 + P2·U1 |
| 3 | P1·U3 | P2·U2 | P3·U1 | P1·U3 + P2·U2 + P3·U1 |
| 4 | — | P2·U3 | P3·U2 | P2·U3 + P3·U2 |
| 5 | — | — | P3·U3 | P3·U3 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
print('Library siap!')

---
## Step 1: Input Data

Masukkan data curah hujan efektif dan ordinat Unit Hydrograph.

In [ ]:
# ── Data DAS ──────────────────────────────────────────────────────────────────
luas_DAS = 250   # km²

# ── Curah hujan efektif per jam (mm) ──────────────────────────────────────────
P = np.array([1, 5, 12, 8, 4, 2, 0], dtype=float)   # 7 jam hujan

# ── Ordinat Unit Hydrograph 1-jam (m³/s/mm) ───────────────────────────────────
U = np.array([0.2, 0.8, 1.5, 1.2, 0.6, 0.3, 0.1], dtype=float)

# Tampilkan data input
print('=== DATA INPUT ===')
jam = np.arange(1, len(P)+1)
df_input = pd.DataFrame({'Jam': jam, 'Hujan Efektif P (mm)': P, 'UH U (m³/s/mm)': U})
df_input.set_index('Jam', inplace=True)
print(df_input.to_string())
print(f'\nTotal hujan efektif : {P.sum():.0f} mm')

---
## Step 2: Hitung Konvolusi UH

Panjang output Q = len(P) + len(U) - 1 = 7 + 7 - 1 = **13 jam**.

Algoritma: untuk setiap waktu `t`, jumlahkan semua perkalian `U[i] * P[t-i]`
yang valid (indeks tidak negatif dan tidak melebihi panjang array).

In [ ]:
n_P = len(P)
n_U = len(U)
n_Q = n_P + n_U - 1   # panjang output

Q = np.zeros(n_Q)

# TODO: Lengkapi algoritma konvolusi di bawah ini
# Petunjuk: gunakan dua nested for-loop
#   - Loop luar : i dari 0 s.d. n_P-1  (indeks hujan)
#   - Loop dalam: j dari 0 s.d. n_U-1  (indeks UH)
#   - Tambahkan kontribusi P[i] * U[j] ke Q[i+j]

for i in range(n_P):
    for j in range(n_U):
        Q[i + j] += P[i] * U[j]   # <-- ini sudah benar, pahami logikanya!

# Verifikasi cepat dengan numpy (boleh digunakan untuk mengecek jawaban)
Q_check = np.convolve(P, U)
print('Apakah hasil loop == numpy convolve?', np.allclose(Q, Q_check))
print(f'\nDebit puncak : {Q.max():.2f} m³/s  pada jam ke-{Q.argmax()+1}')

---
## Step 3: Tabel Hasil

In [ ]:
waktu = np.arange(1, n_Q + 1)

# Hujan efektif per jam (pad dengan nol agar sama panjang dengan Q)
P_padded = np.pad(P, (0, n_Q - n_P))

hasil = pd.DataFrame({
    'Jam'                          : waktu,
    'Hujan Efektif P (mm)'         : P_padded,
    'Debit Limpasan Q (m³/s)'      : Q.round(3)
})
hasil.set_index('Jam', inplace=True)
print(hasil.to_string())

---
## Step 4: Visualisasi Hydrograph

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# ── Hujan efektif (sumbu kanan, dibalik) ─────────────────────────────────────
ax2 = ax1.twinx()
ax2.bar(np.arange(1, n_P+1), P, color='steelblue', alpha=0.4, width=0.7, label='Hujan Efektif P')
ax2.set_ylabel('Hujan Efektif (mm)', color='steelblue', fontsize=11)
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2.invert_yaxis()   # konvensi: hujan di atas, terbalik
ax2.set_ylim(P.max() * 5, 0)

# ── Hydrograph debit (sumbu kiri) ────────────────────────────────────────────
ax1.fill_between(waktu, Q, alpha=0.15, color='tomato')
ax1.plot(waktu, Q, 'tomato', linewidth=2.5, marker='o', markersize=5, label='Debit Q(t)')

# Tandai debit puncak
t_peak  = Q.argmax() + 1
Q_peak  = Q.max()
ax1.annotate(f'Qpeak = {Q_peak:.2f} m³/s\n(jam ke-{t_peak})',
             xy=(t_peak, Q_peak), xytext=(t_peak + 1.5, Q_peak * 0.9),
             arrowprops=dict(arrowstyle='->', color='black'),
             fontsize=10, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

ax1.set_xlabel('Waktu (jam)', fontsize=12)
ax1.set_ylabel('Debit Limpasan Q (m³/s)', fontsize=12)
ax1.set_title(f'Hydrograph Banjir — DAS Progo ({luas_DAS} km²)\nKonvolusi Unit Hydrograph',
              fontsize=13, fontweight='bold')
ax1.set_xlim(0.5, n_Q + 0.5)

# Gabungkan legend dari kedua sumbu
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig('P2.2_hydrograph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai P2.2_hydrograph.png')

---
## Level 2 — Analisis Karakteristik Banjir

In [ ]:
# ── Debit puncak & waktu puncak ───────────────────────────────────────────────
idx_peak = Q.argmax()
Q_peak   = Q[idx_peak]
t_peak   = waktu[idx_peak]

# ── Volume total limpasan ─────────────────────────────────────────────────────
dt_jam    = 1.0        # jam
volume_m3 = np.trapz(Q, waktu) * dt_jam * 3600   # konversi jam → detik
volume_mm = volume_m3 / (luas_DAS * 1e6) * 1000  # konversi ke mm (kedalaman)

# ── Waktu kembali ke baseflow (<5% Qpeak) ────────────────────────────────────
threshold      = 0.05 * Q_peak
t_recession    = waktu[Q > threshold][-1]

print('=' * 45)
print('  RINGKASAN ANALISIS BANJIR')
print('=' * 45)
print(f'  Debit puncak (Qpeak)      : {Q_peak:.3f} m³/s')
print(f'  Waktu puncak (time to peak): jam ke-{t_peak}')
print(f'  Volume total limpasan      : {volume_m3/1e6:.3f} juta m³')
print(f'  Kedalaman limpasan         : {volume_mm:.1f} mm')
print(f'  Waktu resesi (<5% Qpeak)   : jam ke-{t_recession}')

# TODO: Pertanyaan Diskusi
# 1. Mengapa Qpeak tidak terjadi pada jam ke-3 meskipun hujan terbesar di jam ke-3?
# 2. Jika luas DAS digandakan menjadi 500 km², bagaimana pengaruhnya terhadap Q?
# 3. Bagaimana cara mengurangi Qpeak menggunakan pendekatan konservasi lahan?

---
## Level Juara — Uji Sensitivitas: Variasi Skenario Hujan

Bandingkan hydrograph untuk **3 skenario hujan berbeda** pada DAS yang sama.

In [ ]:
skenario = {
    'Hujan Normal (original)' : np.array([1, 5, 12, 8, 4, 2, 0], dtype=float),
    'Hujan Singkat Intens'    : np.array([0, 2, 25, 5, 0, 0, 0], dtype=float),
    'Hujan Merata Lama'       : np.array([4, 4, 4, 4, 4, 4, 4], dtype=float),
}
colors = ['tomato', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(11, 5))

for (nama, P_s), warna in zip(skenario.items(), colors):
    # TODO: hitung Q untuk masing-masing skenario (gunakan np.convolve)
    Q_s = np.convolve(P_s, U)
    t_s = np.arange(1, len(Q_s) + 1)
    ax.plot(t_s, Q_s, color=warna, linewidth=2.2,
            label=f'{nama}  (Qpeak={Q_s.max():.1f} m³/s)')

ax.set_xlabel('Waktu (jam)', fontsize=12)
ax.set_ylabel('Debit (m³/s)', fontsize=12)
ax.set_title('Perbandingan Hydrograph: 3 Skenario Hujan', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# TODO: Diskusikan — skenario mana yang paling berbahaya bagi masyarakat hilir?